
# Session 28 — Fine-Tuning a Small LLaMA with QLoRA

**Computer Vision & AI · One-on-One Series**

In the theory half we walked through how a model goes from a raw next-token predictor to a
helpful assistant, and why updating every weight is out of reach on a normal GPU. This
notebook is the other half: we will actually do it.

### What we are building

| | |
|---|---|
| **Base model** | `TinyLlama/TinyLlama-1.1B-Chat-v1.0` — 1.1B parameters, Apache-2.0, no gated access |
| **Dataset** | `databricks/databricks-dolly-15k` — 15,011 human-written instruction records |
| **Method** | QLoRA — the base model frozen in 4-bit NF4, LoRA adapters trained in bf16/fp16 |
| **Hardware** | One 16 GB T4 (Colab free tier) |
| **Time** | Roughly 15–20 minutes on a 2,000-example subset |

### Before you run anything

In Colab: **Runtime → Change runtime type → Hardware accelerator: T4 GPU**.

Everything else installs from the first cell.

### How to read this notebook

Each section states *what* we are doing and *why it matters*, then shows the code. If you
are short on time, the sections marked **⚙️ Core** are the ones that actually perform the
fine-tune; the others build intuition.


---
## 1. Environment setup

Four libraries do the work here, and it is worth knowing which does what — when something
breaks, the error usually points at one of them.

- **`transformers`** — the model and tokenizer classes.
- **`peft`** — Parameter-Efficient Fine-Tuning. This is what injects the LoRA adapters.
- **`bitsandbytes`** — the CUDA kernels for 4-bit NF4 quantization. This is the *Q* in QLoRA.
- **`trl`** — Transformer Reinforcement Learning. We only use its `SFTTrainer`, a thin
  convenience wrapper around the standard `Trainer` for supervised fine-tuning.
- **`datasets`** — loading and mapping over Dolly.

> **Note on versions.** The PEFT/TRL APIs move quickly. The versions below were chosen to
> work together. If you upgrade one, expect to adjust argument names.

In [ ]:

!pip install -q -U \
    "transformers>=4.44.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.43.0" \
    "trl>=0.9.6" \
    "datasets>=2.20.0" \
    "accelerate>=0.33.0"

print("Installed. If Colab asks you to restart the runtime, do it, then skip this cell.")

In [ ]:
import os, gc, time, json, torch
import transformers, peft, trl, datasets

print(f"torch         {torch.__version__}")
print(f"transformers  {transformers.__version__}")
print(f"peft          {peft.__version__}")
print(f"trl           {trl.__version__}")
print(f"datasets      {datasets.__version__}")
print()

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU."
    )

gpu = torch.cuda.get_device_properties(0)
print(f"GPU           {gpu.name}")
print(f"VRAM          {gpu.total_memory / 1e9:.1f} GB")

# bf16 is nicer than fp16 (wider exponent range, fewer overflow issues) but a T4 is
# Turing-generation and does not support it. Detect and fall back.
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()
print(f"bf16 support  {SUPPORTS_BF16}  ->  training in {'bf16' if SUPPORTS_BF16 else 'fp16'}")

In [ ]:

# A small helper we will reuse to watch memory as we go.

def gpu_report(label=""):
    """Print current and peak GPU allocation."""
    alloc = torch.cuda.memory_allocated() / 1e9
    peak  = torch.cuda.max_memory_allocated() / 1e9
    print(f"{label:<28} allocated {alloc:6.2f} GB   |   peak {peak:6.2f} GB")


def free_memory():
    """Drop cached blocks and reset the peak counter."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


free_memory()
gpu_report("baseline")


---
## 2. Loading the model in 4-bit ⚙️ Core

This is the *Q* in QLoRA. `BitsAndBytesConfig` controls how the frozen base model is
stored, and each argument maps directly onto something from the theory slides:

| Argument | What it does |
|---|---|
| `load_in_4bit=True` | Store base weights in 4 bits instead of 16 |
| `bnb_4bit_quant_type="nf4"` | Use **4-bit NormalFloat**, whose bins are the quantiles of a normal distribution — a better fit for weight distributions than plain INT4 |
| `bnb_4bit_use_double_quant=True` | **Double quantization** — quantize the per-block scaling constants too, saving ~0.37 bits/param |
| `bnb_4bit_compute_dtype` | The dtype weights are *de-quantized into* for each matmul. Storage is 4-bit; arithmetic is not. |

**The point worth pausing on:** the weights are stored in 4 bits, but every forward pass
temporarily de-quantizes a block back to 16-bit to do the actual multiplication. That is
why QLoRA saves memory but costs some speed — you pay compute to buy VRAM.

In [ ]:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
COMPUTE_DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # store the frozen base model in 4 bits
    bnb_4bit_quant_type="nf4",             # 4-bit NormalFloat, not plain int4
    bnb_4bit_use_double_quant=True,        # quantize the quantization constants too
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # de-quantize to this dtype for each matmul
)

free_memory()
t0 = time.time()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},          # everything on GPU 0
    torch_dtype=COMPUTE_DTYPE,
)
model.config.use_cache = False   # incompatible with gradient checkpointing; re-enable later

print(f"Loaded in {time.time() - t0:.1f}s")
gpu_report("after 4-bit load")


### Did the quantization actually help?

Let's check rather than assume. We compute what the same weights *would* occupy in fp16
and compare against what the 4-bit model actually uses.

In [ ]:

n_params = sum(p.numel() for p in model.parameters())

fp16_gb = n_params * 2 / 1e9        # 2 bytes per parameter
fp32_gb = n_params * 4 / 1e9        # 4 bytes per parameter
actual_gb = torch.cuda.memory_allocated() / 1e9

print(f"Parameters                {n_params/1e9:.2f} B")
print(f"Would need in fp32        {fp32_gb:.2f} GB")
print(f"Would need in fp16        {fp16_gb:.2f} GB")
print(f"Actually using (NF4)      {actual_gb:.2f} GB")
print(f"Reduction vs fp16         {fp16_gb/actual_gb:.1f}x")
print()
print("Note: this is roughly 4x rather than exactly 4x — LayerNorms, embeddings and")
print("the LM head are kept in higher precision, because quantizing them hurts quality")
print("far more than it saves memory.")


### The tokenizer

Two housekeeping steps that cause confusing errors if you skip them:

1. **A pad token.** LLaMA-family tokenizers ship without one. Batching requires padding, so
   we reuse the EOS token. (Padding tokens are masked out of the loss, so this is safe.)
2. **Right padding.** For *training* we pad on the right. Left padding is the convention for
   batched *generation*. Mixing them up produces a model that trains fine and generates
   nonsense.

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size    {len(tokenizer)}")
print(f"EOS token     {tokenizer.eos_token!r}  (id {tokenizer.eos_token_id})")
print(f"PAD token     {tokenizer.pad_token!r}  (id {tokenizer.pad_token_id})")
print(f"Padding side  {tokenizer.padding_side}")


---
## 3. Baseline — what does the model do *before* we touch it?

This step is easy to skip and worth not skipping. If you do not record the baseline now,
you have no honest way to claim the fine-tune helped later.

TinyLlama-1.1B-**Chat** has already been instruction-tuned by its authors, so it will not
be incoherent. What we are looking for after our fine-tune is a shift toward Dolly's
particular register: shorter, flatter, more encyclopaedic answers.

In [ ]:

def generate(prompt, max_new_tokens=120, temperature=0.7, model=model):
    """Format a prompt with the chat template and generate a response."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    model.config.use_cache = True
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    model.config.use_cache = False

    # Slice off the prompt so we only return what the model added.
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                            skip_special_tokens=True).strip()


TEST_PROMPTS = [
    "When did Virgin Australia start operating?",
    "Why can camels survive for long without water?",
    "Give me a short list of the largest deserts in the world.",
    "What is the difference between a violin and a viola?",
]

baseline_outputs = {}
for p in TEST_PROMPTS:
    baseline_outputs[p] = generate(p)
    print("=" * 78)
    print(f"PROMPT   {p}")
    print("-" * 78)
    print(baseline_outputs[p])
print("=" * 78)


---
## 4. The dataset ⚙️ Core

`databricks-dolly-15k` is 15,011 instruction records written by Databricks employees,
released under CC BY-SA 3.0. Each record has four fields:

- `instruction` — what the user asked
- `context` — optional supporting passage (present in about a third of records)
- `response` — the answer a human wrote
- `category` — one of eight task types (`open_qa`, `closed_qa`, `summarization`, …)

It is a good teaching dataset precisely because it is small and hand-written: you can read
the examples yourself and form a view about what the model is being taught.

In [ ]:

from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

print(dataset)
print()
print("Category distribution:")
from collections import Counter
for cat, n in Counter(dataset["category"]).most_common():
    print(f"  {cat:<22} {n:>6}  ({n/len(dataset)*100:.1f}%)")

In [ ]:

# Read a couple of real records before we transform anything.
for i in [0, 1]:
    r = dataset[i]
    print("=" * 78)
    print(f"CATEGORY     {r['category']}")
    print(f"INSTRUCTION  {r['instruction']}")
    ctx = r["context"]
    print(f"CONTEXT      {(ctx[:200] + '…') if len(ctx) > 200 else (ctx or '(none)')}")
    print(f"RESPONSE     {r['response'][:300]}")
print("=" * 78)


### Formatting into a chat template

The model was pre-trained and chat-tuned with a specific prompt format. If we invent our
own, we fight against that formatting rather than building on it — so we use the
tokenizer's built-in `chat_template`, which for TinyLlama looks like:

```
<|user|>
{the instruction, plus context if present}</s>
<|assistant|>
{the response}</s>
```

**Why this matters more than it looks:** the special tokens `<|user|>` and `<|assistant|>`
are how the model knows whose turn it is. Get them wrong and the model never learns to stop
talking — it will happily continue the conversation on your behalf.

In [ ]:

def format_record(record):
    """Turn one Dolly record into a single chat-formatted training string."""
    instruction = record["instruction"].strip()
    context = (record["context"] or "").strip()

    # Fold the optional context into the user turn.
    user_content = f"{instruction}\n\n{context}" if context else instruction

    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": record["response"].strip()},
    ]
    # tokenize=False -> return the formatted string; SFTTrainer tokenizes for us.
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}


formatted = dataset.map(format_record, remove_columns=dataset.column_names)

print("One fully formatted training example:")
print("=" * 78)
print(formatted[0]["text"])
print("=" * 78)


### Trimming to fit the session

Two practical decisions, both about keeping the live demo inside 20 minutes:

1. **Drop very long examples.** A handful of Dolly records have multi-paragraph contexts.
   They dominate the memory budget and teach us nothing extra, so we cap length.
2. **Subsample to 2,000 records.** Enough to see a real behavioural shift; short enough to
   finish while we watch.

For a real project you would use the full set and train for 2–3 epochs. The code is
identical — only `N_SAMPLES` and `num_train_epochs` change.

In [ ]:

MAX_SEQ_LENGTH = 512
N_SAMPLES = 2000
SEED = 42

# Filter by token count, not characters — characters are a poor proxy.
def short_enough(example):
    return len(tokenizer(example["text"])["input_ids"]) <= MAX_SEQ_LENGTH

before = len(formatted)
filtered = formatted.filter(short_enough)
print(f"Kept {len(filtered)} of {before} records "
      f"({len(filtered)/before*100:.1f}%) under {MAX_SEQ_LENGTH} tokens")

train_dataset = filtered.shuffle(seed=SEED).select(range(min(N_SAMPLES, len(filtered))))

# Hold out a small slice so we can watch for overfitting.
split = train_dataset.train_test_split(test_size=0.05, seed=SEED)
train_dataset, eval_dataset = split["train"], split["test"]

print(f"Train  {len(train_dataset)}")
print(f"Eval   {len(eval_dataset)}")

lengths = [len(tokenizer(t)["input_ids"]) for t in train_dataset["text"][:500]]
print(f"\nToken length over a 500-sample probe: "
      f"mean {sum(lengths)/len(lengths):.0f}, max {max(lengths)}")


---
## 5. Attaching the LoRA adapters ⚙️ Core

Now the *LoRA* half. Two steps:

**`prepare_model_for_kbit_training`** does the unglamorous plumbing that makes a quantized
model trainable — casts LayerNorms to fp32 for numerical stability, enables gradient
checkpointing, and makes the input embeddings require gradients so that gradients can flow
back through the frozen stack.

**`LoraConfig`** is where the four knobs from the slides appear:

| Parameter | Value | Reasoning |
|---|---|---|
| `r` | 16 | Rank of the update. 8–16 suits format and style adaptation, which is what Dolly teaches. |
| `lora_alpha` | 32 | The update is scaled by `alpha / r`. The `alpha = 2r` convention gives a scale of 2. |
| `lora_dropout` | 0.05 | Light regularisation — our training set is small. |
| `target_modules` | all 7 linear projections | The QLoRA paper's finding: adapting *every* linear layer, not just attention, is what closes the gap to full fine-tuning. |

The seven modules are the four attention projections (`q_proj`, `k_proj`, `v_proj`,
`o_proj`) and the three MLP projections (`gate_proj`, `up_proj`, `down_proj`).

In [ ]:

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",                 # do not train bias terms
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # attention
        "gate_proj", "up_proj", "down_proj",         # MLP
    ],
)

model = get_peft_model(model, lora_config)
free_memory()
gpu_report("after attaching adapters")


### How little are we actually training?

This is the number that makes the whole approach make sense — worth reading out loud in
the session.

In [ ]:

trainable, total = 0, 0
for _, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print(f"Total parameters      {total:>14,}")
print(f"Trainable parameters  {trainable:>14,}")
print(f"Trainable share       {trainable/total*100:>13.3f}%")
print()
print(f"Adapter size on disk  ~{trainable * 2 / 1e6:.0f} MB (fp16)")
print(f"Full checkpoint       ~{total * 2 / 1e9:.1f} GB (fp16)")
print()
print("This is the deployment story: one shared base model, plus a small adapter")
print("per task, instead of a full copy of the model for every task.")

In [ ]:

# Sanity check: confirm the base weights really are frozen and only LoRA is live.
lora_params = [n for n, p in model.named_parameters() if p.requires_grad]
print(f"{len(lora_params)} trainable tensors, all LoRA. First five:")
for n in lora_params[:5]:
    print("  ", n)

assert all("lora" in n.lower() for n in lora_params), \
    "Something other than a LoRA adapter is trainable!"
print("\nConfirmed: every trainable tensor is a LoRA adapter.")


---
## 6. Training ⚙️ Core

A note on the two batch-size arguments, because they confuse everyone the first time:

- `per_device_train_batch_size=4` — how many examples go through the GPU at once. Bounded
  by VRAM.
- `gradient_accumulation_steps=4` — how many of those batches we accumulate gradients over
  before stepping the optimizer.

**Effective batch size = 4 × 4 = 16.** Gradient accumulation buys you the training dynamics
of a large batch with the memory of a small one. This is the standard trick for fitting
real training onto a small card.

`optim="paged_adamw_8bit"` is the third QLoRA innovation: an 8-bit Adam whose state pages
out to CPU RAM during memory spikes instead of crashing the run.

In [ ]:

import inspect
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "tinyllama-dolly-qlora"


def keep_supported(cls, kwargs, aliases=()):
    """Drop any kwargs this version of `cls` does not accept.

    The TRL/PEFT APIs get renamed fairly often (max_seq_length -> max_length,
    tokenizer -> processing_class, ...). Filtering against the actual signature
    keeps this notebook working across versions instead of dying on a TypeError.

    `aliases` is a list of (preferred, deprecated) pairs. If both names are
    still accepted we keep only the preferred one, so we never pass the same
    value twice under two spellings.
    """
    params = inspect.signature(cls.__init__).parameters
    if any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values()):
        # Signature is (**kwargs) — we cannot introspect it, so pass everything.
        allowed = set(kwargs)
    else:
        allowed = set(params)
    out = {k: v for k, v in kwargs.items() if k in allowed}

    for preferred, deprecated in aliases:
        if preferred in out and deprecated in out:
            out.pop(deprecated)

    dropped = sorted(set(kwargs) - set(out))
    if dropped:
        print(f"[compat] {cls.__name__}: not using {', '.join(dropped)}")
    return out


config_kwargs = dict(
    output_dir=OUTPUT_DIR,

    # --- schedule -------------------------------------------------------
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch size = 16
    max_steps=-1,                         # set a positive number for a quick smoke test

    # --- optimizer ------------------------------------------------------
    optim="paged_adamw_8bit",             # QLoRA's paged optimizer
    learning_rate=2e-4,                   # ~10x higher than full fine-tuning: adapters
                                          # start at zero and must move meaningfully
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_grad_norm=0.3,
    weight_decay=0.001,

    # --- precision ------------------------------------------------------
    bf16=SUPPORTS_BF16,
    fp16=not SUPPORTS_BF16,

    # --- memory ---------------------------------------------------------
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # --- data -----------------------------------------------------------
    max_seq_length=MAX_SEQ_LENGTH,        # newer TRL calls this `max_length`
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,                        # one example per sequence — clearer to teach

    # --- logging --------------------------------------------------------
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    report_to="none",                     # no W&B prompt mid-session
    seed=SEED,
)

sft_config = SFTConfig(**keep_supported(
    SFTConfig, config_kwargs, aliases=[("max_length", "max_seq_length")]
))

trainer_kwargs = dict(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,           # older TRL calls this `tokenizer`
    tokenizer=tokenizer,
)

trainer = SFTTrainer(**keep_supported(
    SFTTrainer, trainer_kwargs, aliases=[("processing_class", "tokenizer")]
))

steps = len(train_dataset) // (4 * 4)
print(f"\nAbout {steps} optimizer steps over 1 epoch.")
print("Expect roughly 15-20 minutes on a T4.")

In [ ]:

free_memory()
t0 = time.time()

train_result = trainer.train()

elapsed = time.time() - t0
print()
print(f"Finished in {elapsed/60:.1f} minutes")
print(f"Final training loss  {train_result.training_loss:.4f}")
gpu_report("peak during training")


### Reading the loss curve

Two things to check, and they matter more than the absolute numbers:

1. **Training loss should fall and then flatten.** A curve that is still dropping steeply at
   the end means you stopped too early — train longer or raise `r`.
2. **Eval loss should track training loss.** If training loss keeps falling while eval loss
   turns upward, the adapter is memorising. Respond with more data, more dropout, or fewer
   epochs — in that order of preference.

Absolute loss values are not comparable across datasets or tokenizers, so resist the urge
to compare this number to one from a different run.

In [ ]:

import matplotlib.pyplot as plt

history = trainer.state.log_history
train_pts = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_pts  = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(9, 4.5))
if train_pts:
    ax.plot(*zip(*train_pts), label="train loss", linewidth=1.8, color="#02C39A")
if eval_pts:
    ax.plot(*zip(*eval_pts), label="eval loss", linewidth=1.8,
            color="#F96167", marker="o", markersize=4)

ax.set_xlabel("optimizer step")
ax.set_ylabel("loss")
ax.set_title("QLoRA fine-tuning — TinyLlama-1.1B on Dolly-15k")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


---
## 7. Saving the adapter

Notice what gets written to disk: only the adapter weights and a small config recording
which base model they belong to. That is the whole deployment argument — you ship megabytes,
not gigabytes.

In [ ]:

ADAPTER_DIR = f"{OUTPUT_DIR}/final-adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Contents of {ADAPTER_DIR}:\n")
total_bytes = 0
for f in sorted(os.listdir(ADAPTER_DIR)):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    total_bytes += size
    print(f"  {f:<34} {size/1e6:>8.2f} MB")

print(f"\n  {'TOTAL':<34} {total_bytes/1e6:>8.2f} MB")
print(f"\nCompare with a full fp16 checkpoint of the same model: ~{total*2/1e9:.1f} GB")

In [ ]:

# The adapter config records its parent model, which is how PEFT reassembles things later.
with open(os.path.join(ADAPTER_DIR, "adapter_config.json")) as f:
    print(json.dumps(json.load(f), indent=2))


---
## 8. Before and after

The honest test. We run the same four prompts we ran in section 3 and put the outputs
side by side.

**Set expectations before you look.** TinyLlama-Chat was already instruction-tuned, and we
trained for one epoch on 2,000 examples. You should not expect a dramatic capability jump.
What you *should* see is a shift in register toward Dolly's house style: more direct,
shorter, more factual, less conversational padding.

If you see no difference at all, that is informative too — it usually means the learning
rate was too low or the dataset too small for the adaptation to take.

In [ ]:

tuned_outputs = {}
for p in TEST_PROMPTS:
    tuned_outputs[p] = generate(p, model=trainer.model)

for p in TEST_PROMPTS:
    print("=" * 78)
    print(f"PROMPT   {p}")
    print("=" * 78)
    print("--- BEFORE (base TinyLlama-Chat) " + "-" * 44)
    print(baseline_outputs[p])
    print()
    print("--- AFTER (QLoRA fine-tuned on Dolly) " + "-" * 39)
    print(tuned_outputs[p])
    print()


### A more rigorous check

Eyeballing four prompts is a demo, not an evaluation. For a real project you would want at
least:

- **A held-out test set** the model never saw, scored with perplexity.
- **Task-specific metrics** — exact match or F1 for QA, ROUGE for summarization.
- **Pairwise human or LLM-judge comparison** on a fixed prompt set. This is what the
  Guanaco Elo numbers from the slides are.

The cell below does the cheapest of these — perplexity on our held-out slice — so you can
see the shape of a real evaluation.

In [ ]:

import math

eval_metrics = trainer.evaluate()
loss = eval_metrics["eval_loss"]

print(f"Held-out eval loss   {loss:.4f}")
print(f"Perplexity           {math.exp(loss):.2f}")
print()
print("Perplexity is exp(cross-entropy loss): roughly 'how many tokens is the model")
print("effectively choosing between at each step'. Lower is better, but the number is")
print("only meaningful compared against another run on THIS dataset and tokenizer.")


---
## 9. Reloading the adapter from scratch

This is what deployment actually looks like: load the base model once, then attach whichever
adapter you need. Swapping tasks means swapping a 30 MB file, not reloading a 2 GB model.

We restart the runtime state by deleting our objects first, so this genuinely demonstrates
loading from disk rather than reusing what is already in memory.

In [ ]:

del trainer, model
free_memory()
gpu_report("after clearing memory")

In [ ]:

from peft import PeftModel

# Step 1: the base model, quantized exactly as before.
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
)

# Step 2: attach the adapter we just trained.
reloaded = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
reloaded.eval()

gpu_report("base + adapter reloaded")

print()
print(generate("Why can camels survive for long without water?", model=reloaded))


---
## 10. Merging the adapter for deployment

`PeftModel` computes `W₀x + BAx` as two separate operations, which adds a small amount of
overhead at inference time. Once you are finished training you can **fold `BA` into `W₀`**,
producing an ordinary model with no LoRA machinery at all — this is the "zero inference
latency" claim from the slides.

**The one catch:** merging requires 16-bit weights. You cannot cleanly merge into a 4-bit
base, so the merge is done against an fp16 copy of the base model. That means merging needs
more RAM than training did.

> ⚠️ **On a free T4 this cell may run out of memory.** That is expected and is itself worth
> discussing — merging is a deployment-time step, usually done on a bigger machine. The code
> is here so you know the shape of it; skip it if the GPU complains.

In [ ]:

MERGE = False   # flip to True if you have the headroom (or are on CPU with enough RAM)

if MERGE:
    del base_model, reloaded
    free_memory()

    # Reload the base model in fp16 — NOT quantized — so the merge is lossless.
    fp16_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map={"": 0},
    )
    merged = PeftModel.from_pretrained(fp16_base, ADAPTER_DIR)
    merged = merged.merge_and_unload()      # fold BA into W0

    MERGED_DIR = f"{OUTPUT_DIR}/merged-fp16"
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)

    size = sum(
        os.path.getsize(os.path.join(MERGED_DIR, f))
        for f in os.listdir(MERGED_DIR)
    )
    print(f"Merged model saved to {MERGED_DIR}  ({size/1e9:.2f} GB)")
    print("This is now a plain causal LM — no PEFT needed to load it.")
else:
    print("Skipped. Set MERGE = True to run the merge.")
    print()
    print("When you would actually do this:")
    print("  - deploying to a serving stack that does not know about PEFT")
    print("  - exporting to GGUF / ONNX / TensorRT")
    print("  - squeezing out the last few percent of inference latency")
    print()
    print("When you would NOT:")
    print("  - you want to keep swapping between several task adapters")
    print("  - you want to keep training the adapter later")


---
## 11. Things to try yourself

In rough order of effort. Each one is a single-parameter change to the code above.

1. **Change the rank.** Re-run with `r=8` and `r=64`. Compare trainable parameter count,
   peak memory, wall-clock time and final eval loss. Does more capacity actually help on
   2,000 examples, or does it just overfit?

2. **Attention-only adapters.** Set `target_modules=["q_proj", "v_proj"]` — the original
   LoRA paper's configuration. You will train far fewer parameters. Is the result
   noticeably worse? This is the QLoRA paper's claim, tested on your own data.

3. **Turn off double quantization.** Set `bnb_4bit_use_double_quant=False` and measure the
   memory difference. On a 1.1B model it will be small — extrapolate to 65B.

4. **Compare NF4 against FP4.** Change `bnb_4bit_quant_type` to `"fp4"` and compare eval
   loss. This directly tests the paper's central claim about NF4.

5. **Train on one category.** Filter Dolly to `category == "summarization"` and fine-tune
   on that alone. A narrow adapter should show a much more visible behavioural shift than
   our general one did.

6. **Swap the dataset.** Try `yahma/alpaca-cleaned` or `sahil2801/CodeAlpaca-20k`. Only the
   `format_record` function needs to change — everything downstream is dataset-agnostic.

7. **Scale up the base model.** `Qwen/Qwen2.5-1.5B-Instruct` or a 3B model will still fit in
   4-bit on a T4. Where does it stop fitting?


---
## 12. Recap

What we did, in one paragraph: we loaded a 1.1B-parameter causal LM with its weights
compressed to 4-bit NormalFloat, froze all of it, attached rank-16 LoRA adapters to every
linear layer, and trained only those adapters — about 0.5% of the parameters — on 2,000
human-written instruction records formatted with the model's own chat template. Then we
saved a ~30 MB adapter, reloaded it against a fresh base model, and compared outputs.

**The three ideas worth carrying forward:**

- **Fine-tuning is reshaping, not teaching.** We changed how the model answers, not what it
  knows. When you need new facts, reach for RAG.
- **The optimizer is the memory hog, not the weights.** LoRA wins by shrinking what the
  optimizer has to track, and QLoRA adds a smaller frozen base on top of that.
- **Adapters are artifacts you can manage.** Small, versionable, swappable, composable — a
  much better operational story than a folder of near-identical multi-gigabyte checkpoints.

---

### References

- Hu et al., 2021 — *LoRA: Low-Rank Adaptation of Large Language Models*, [arXiv:2106.09685](https://arxiv.org/abs/2106.09685)
- Dettmers et al., 2023 — *QLoRA: Efficient Finetuning of Quantized LLMs*, [arXiv:2305.14314](https://arxiv.org/abs/2305.14314)
- Ouyang et al., 2022 — *Training language models to follow instructions with human feedback*, [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
- Rafailov et al., 2023 — *Direct Preference Optimization*, [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
- Devlin et al., 2019 — *BERT*, [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
- [TinyLlama-1.1B-Chat-v1.0](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) · [databricks-dolly-15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k)
- [Hugging Face PEFT documentation](https://huggingface.co/docs/peft) · [TRL documentation](https://huggingface.co/docs/trl)